<a href="https://colab.research.google.com/github/temariid/ColabFilesSessia1/blob/main/zadanie2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

import plotly
import plotly.graph_objects as go
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson

df = pd.read_csv('/content/multiregress-092022.csv', decimal='.', sep=',')
df
df = df.drop('i', axis = 1)

df

df.corr(method='pearson')

y = df.tc.to_numpy()
x= df[['ne', 'Me']].to_numpy()
X=sm.add_constant(x)
print('y:\n', y, '\n\n X: \n', X, '\n')

Xt=X.transpose()
B=np.linalg.inv(Xt.dot(X))
print('B:', B, '\n')
a=np.dot(B.dot(Xt), y)
print('Параметры регрессии: a0:', a[0], 'a1:', a[1], 'a2:', a[2], '\n')

y_r=X.dot(a)
print('Рассчетные значения y \n', y_r, '\n')

e=y-y_r
print('Остатки е \n', e)

y=df.tc
X=df[['ne', 'Me']]
X=sm.add_constant(X)
print('y:\n', y, '\n\n X: \n', X, '\n')

model = sm.OLS(y,X)
reg=model.fit()

print("Параметры регрессии: ", reg.params, "\n")
a0=reg.params['const']
a1=reg.params['ne']
a2=reg.params['Me']

y_r = reg.fittedvalues
e=reg.resid
print("Расчетные(Прогнозные) значения:\n", y_r)
print("Остатки:\n",e)


print("Отчет по регрессии")
print('Тип объекта: ', type(reg))
print(reg.summary(), '\n\n')

print("t-статистика:\n", reg.tvalues, "\n\n")
print("Расчетное значение критерия Дарбина-Уотсона:\n", durbin_watson(reg.resid), "\n\n")
print("Первый коэффициент автокорреляции остатков:", reg.resid.autocorr(lag=1), "\n\n")

alpha = 0.05
print('Уровень ошибки', alpha,
      '\nУровень значимости слоя двустороннего обратного распределения Стьюдента:', 1-alpha/2,
      '\nУровень значимости для обратного F-распределения:', 1-alpha)
print('Число степеней свободы регрессии:', reg.df_model)
print('Число степеней свободы остатков:', reg.df_resid)

t_inv=stats.t.ppf(1-alpha/2, reg.df_resid) #квантиль t-распределения
print('Табличное значение критерия Стьюдента', t_inv)

F_inv=stats.f.ppf(1-alpha, reg.df_model, reg.df_resid) #квантиль F-распределения
print('Табличное значение критерия Фишера', F_inv)

fig = go.Figure()
fig.add_trace(
    go.Scatter3d(
        x=df['ne'],
        y=df['Me'],
        z=df['tc'],
        mode='markers',
        name='Исходные данные',
        marker=dict(
            color='rgba(0,0,255,1)',
            size = 7,
            opacity=0.8,
        ),
        showlegend=True

    )
)
fig.add_trace(
    go.Scatter3d(
        x=df['ne'],
        y=df['Me'],
        z=y_r,
        mode='markers',
        name='лин. модель',
        marker=dict(
            color='rgba(0,255,0,1)',
            size = 7,
            opacity=0.8,
        ),
        showlegend=True

    )
)
fig.update_layout(scene=dict(
    xaxis_title="ne, об/мин",
    yaxis_title="Me, Hm",
    zaxis_title='t, C'),
                      title_text="Температура в зависимости от частоты и крутящего момента",
    title_font_size=20,
                  width = 700,
                  margin =dict(r=20,b=10,l=10,t=10))
fig.show()

x_pr=np.array([1,1000,250])
y_pr=reg.predict(x_pr)
print("Точечный прогноз: y=", y_pr[0], '\n')

Se=np.sqrt(reg.scale)

alpha = 0.01
t_inv=stats.t.ppf(1-alpha/2, reg.df_resid)
print('Табличное значение критерия Стьюдента', t_inv)
U =Se*t_inv*np.sqrt(1+np.dot(x_pr.transpose(), B.dot(x_pr)))
print('Ширина доверительного интервала', U, 'при уровне ошибки', alpha)
print('y=', y_pr[0]-U, '...',y_pr[0]+U)

fig.add_trace(
    go.Scatter3d(
        x=[x_pr[1]],
        y=[x_pr[2]],
        z=[y_pr[0]],
        error_z=dict(
            type ='data',
            array=[U],
            visible=True),
        mode='markers',
        name='прогноз',
        marker=dict(
            color='rgba(255,0,0,1)',
            size=7,
            opacity=0.8,
        ),
        showlegend=True
    )
)
fig.update_layout(scene=dict(
    xaxis_title="ne, об/мин",
    yaxis_title="Me, Hm",
    zaxis_title='t, C'),
                      title_text="Температура в зависимости от частоты и крутящего момента",
    title_font_size=20,
                  width = 700,
                  margin =dict(r=20,b=10,l=10,t=10))
fig.show()

#print(f"Уравнение регрессии: tc = {a0:.3f} + {a1:.5f} * ne + {a2:.5f} * Me")

y:
 [80.7 82.8 83.2 79.3 80.2 80.7 83.2 83.7 80.3 80.4 81.5 82.3 84.3 80.2
 80.7 82.1 83.1 85.  80.7 80.8 82.8 83.6] 

 X: 
 [[1.00e+00 1.20e+03 1.40e+02]
 [1.00e+00 1.20e+03 2.10e+02]
 [1.00e+00 1.20e+03 2.52e+02]
 [1.00e+00 1.40e+03 2.80e+01]
 [1.00e+00 1.40e+03 7.00e+01]
 [1.00e+00 1.40e+03 1.40e+02]
 [1.00e+00 1.40e+03 2.10e+02]
 [1.00e+00 1.40e+03 2.52e+02]
 [1.00e+00 1.60e+03 2.80e+01]
 [1.00e+00 1.60e+03 7.00e+01]
 [1.00e+00 1.60e+03 1.40e+02]
 [1.00e+00 1.60e+03 2.10e+02]
 [1.00e+00 1.60e+03 2.52e+02]
 [1.00e+00 1.80e+03 2.80e+01]
 [1.00e+00 1.80e+03 7.00e+01]
 [1.00e+00 1.80e+03 1.40e+02]
 [1.00e+00 1.80e+03 2.10e+02]
 [1.00e+00 1.80e+03 2.52e+02]
 [1.00e+00 2.00e+03 2.80e+01]
 [1.00e+00 2.00e+03 7.00e+01]
 [1.00e+00 2.00e+03 1.40e+02]
 [1.00e+00 2.00e+03 2.10e+02]] 

B: [[ 2.31825787e+00 -1.22720430e-03 -2.00419051e-03]
 [-1.22720430e-03  7.06837052e-07  5.82570004e-07]
 [-2.00419051e-03  5.82570004e-07  7.41355527e-06]] 

Параметры регрессии: a0: 76.17962961022461 a1: 0.0018

Точечный прогноз: y= 82.78664734580114 

Табличное значение критерия Стьюдента 2.860934606449914
Ширина доверительного интервала 1.4052147524587841 при уровне ошибки 0.01
y= 81.38143259334235 ... 84.19186209825993


Уравнение регрессии: tc = 76.180 + 0.00184 * ne + 0.01905 * Me
